# Preference Signal Detector — V2.1 targeted retraining — Dual GPU reviewed

This notebook retrains `distilbert-base-multilingual-cased` from the base model
using the combined V2.1 dataset.

### Improvements in this reviewed version

- automatically detects **1 or 2 Kaggle GPUs**;
- uses `torch.nn.DataParallel` when 2 GPUs are available;
- keeps the same effective training batch size:
  - 1 GPU: batch 16 × grad accumulation 2 = 32
  - 2 GPUs: batch 32 × grad accumulation 1 = 32
- handles the DataParallel multi-GPU loss correctly with `.mean()`;
- unwraps the model before `save_pretrained()`;
- re-wraps the best checkpoint on both GPUs for evaluation;
- pre-tokenizes datasets in RAM once, instead of tokenizing every sample again
  at every epoch;
- keeps `MAX_LENGTH=128`;
- keeps `NUM_WORKERS=0` to avoid Kaggle DataLoader worker shutdown warnings;
- does **not** load or evaluate the fresh final holdout V3.

Scientific protocol:

1. TRAIN V2.1 = 28,400
2. VALIDATION V2.1 = 5,800
3. select the best checkpoint on VALIDATION macro-F1 at threshold 0.5
4. calibrate a raw Transformer threshold candidate on VALIDATION only
5. freeze checkpoint + raw threshold candidate
6. evaluate only already-inspected regression sets:
   - old V2 validation (5,200)
   - old V2 holdout (1,200)
   - legacy V1 test (4,000)
7. export the V2.1 artifact

The fresh final holdout V3 is **not used or loaded anywhere in this notebook**.


In [13]:
!pip install -q "transformers>=4.44,<5" "accelerate>=0.34" "scikit-learn>=1.5" safetensors


In [14]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import time
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

SEED = 20260823
BASE_MODEL = "distilbert-base-multilingual-cased"
MAX_LENGTH = 128

# Preserve the same effective optimizer batch size = 32.
SINGLE_GPU_BATCH_SIZE = 16
SINGLE_GPU_GRAD_ACCUM = 2

MAX_EPOCHS = 5
PATIENCE = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
TARGET_RECALL = 0.985

# We pre-tokenize everything once, so workers are not needed.
NUM_WORKERS = 0
TOKENIZE_BATCH_SIZE = 2048

WORKING = Path("/kaggle/working")
OUT = WORKING / "preference_signal_detector_v2_1"
BEST_DIR = WORKING / "best_v2_1_checkpoint"
FINAL_ZIP = WORKING / "preference_signal_detector_v2_1.zip"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GPU_COUNT = torch.cuda.device_count()

if GPU_COUNT == 0:
    DEVICE = torch.device("cpu")
    USE_MULTI_GPU = False
    TRAIN_BATCH_SIZE = SINGLE_GPU_BATCH_SIZE
    GRAD_ACCUM = SINGLE_GPU_GRAD_ACCUM
    EVAL_BATCH_SIZE = SINGLE_GPU_BATCH_SIZE * 2

elif GPU_COUNT == 1:
    DEVICE = torch.device("cuda:0")
    USE_MULTI_GPU = False
    TRAIN_BATCH_SIZE = SINGLE_GPU_BATCH_SIZE
    GRAD_ACCUM = SINGLE_GPU_GRAD_ACCUM
    EVAL_BATCH_SIZE = SINGLE_GPU_BATCH_SIZE * 2

else:
    # Kaggle dual-GPU runtime: use GPU 0 + GPU 1.
    DEVICE = torch.device("cuda:0")
    USE_MULTI_GPU = True

    # Total batch presented to DataParallel. It will be split across GPUs.
    TRAIN_BATCH_SIZE = SINGLE_GPU_BATCH_SIZE * 2
    GRAD_ACCUM = 1
    EVAL_BATCH_SIZE = SINGLE_GPU_BATCH_SIZE * 4

print("=" * 80)
print("RUNTIME")
print("=" * 80)
print("torch:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("GPU_COUNT:", GPU_COUNT)

for index in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(index)
    print(
        f"GPU {index}: {torch.cuda.get_device_name(index)} | "
        f"{props.total_memory / (1024**3):.2f} GiB"
    )

print("USE_MULTI_GPU:", USE_MULTI_GPU)
print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)
print("GRAD_ACCUM:", GRAD_ACCUM)
print("EFFECTIVE_BATCH_SIZE:", TRAIN_BATCH_SIZE * GRAD_ACCUM)
print("EVAL_BATCH_SIZE:", EVAL_BATCH_SIZE)

if GPU_COUNT >= 2:
    assert TRAIN_BATCH_SIZE * GRAD_ACCUM == 32
    print("DUAL GPU CONFIG: PASS — GPU 0 + GPU 1 will be used.")
elif GPU_COUNT == 1:
    assert TRAIN_BATCH_SIZE * GRAD_ACCUM == 32
    print("SINGLE GPU FALLBACK: PASS.")
else:
    print("WARNING: no GPU detected; training will be very slow.")


RUNTIME
torch: 2.10.0+cu128
cuda version: 12.8
GPU_COUNT: 2
GPU 0: Tesla T4 | 14.56 GiB
GPU 1: Tesla T4 | 14.56 GiB
USE_MULTI_GPU: True
TRAIN_BATCH_SIZE: 32
GRAD_ACCUM: 1
EFFECTIVE_BATCH_SIZE: 32
EVAL_BATCH_SIZE: 64
DUAL GPU CONFIG: PASS — GPU 0 + GPU 1 will be used.


In [15]:
def find_file(filename: str) -> Path:
    candidates = list(Path("/kaggle/input").rglob(filename))

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename!r} under /kaggle/input, "
            f"found {len(candidates)}: {candidates}"
        )

    return candidates[0]


paths = {
    "train": find_file("preference_signal_train_v2_1.jsonl"),
    "val": find_file("preference_signal_val_v2_1.jsonl"),
    "regression_val_v2": find_file("preference_signal_regression_val_v2_5200.jsonl"),
    "regression_holdout_v2": find_file("preference_signal_regression_holdout_v2_1200.jsonl"),
    "legacy_test": find_file("preference_signal_legacy_test_v1_4000.jsonl"),
}

print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))


def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []

    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


train_df = load_jsonl(paths["train"])
val_df = load_jsonl(paths["val"])
reg_val_df = load_jsonl(paths["regression_val_v2"])
reg_holdout_df = load_jsonl(paths["regression_holdout_v2"])
legacy_df = load_jsonl(paths["legacy_test"])

for name, frame in {
    "train": train_df,
    "val": val_df,
    "reg_val": reg_val_df,
    "reg_holdout": reg_holdout_df,
    "legacy": legacy_df,
}.items():
    assert {"text", "label_id"}.issubset(frame.columns), (
        name,
        frame.columns.tolist(),
    )

assert len(train_df) == 28400, len(train_df)
assert len(val_df) == 5800, len(val_df)
assert len(reg_val_df) == 5200, len(reg_val_df)
assert len(reg_holdout_df) == 1200, len(reg_holdout_df)
assert len(legacy_df) == 4000, len(legacy_df)

assert train_df.label_id.value_counts().to_dict() in (
    {0: 14200, 1: 14200},
    {1: 14200, 0: 14200},
)

assert val_df.label_id.value_counts().to_dict() in (
    {0: 2900, 1: 2900},
    {1: 2900, 0: 2900},
)

assert train_df["text"].nunique() == 28400
assert val_df["text"].nunique() == 5800
assert set(train_df["text"]).isdisjoint(set(val_df["text"]))

if "group_id" in train_df.columns and "group_id" in val_df.columns:
    assert set(train_df["group_id"]).isdisjoint(set(val_df["group_id"]))

assert "is_targeted_v21" in train_df.columns
assert "is_targeted_v21" in val_df.columns

train_targeted_mask = train_df["is_targeted_v21"].fillna(False).astype(bool)
val_targeted_mask = val_df["is_targeted_v21"].fillna(False).astype(bool)

targeted_train = train_df.loc[train_targeted_mask].copy()
targeted_val = val_df.loc[val_targeted_mask].copy()

assert len(targeted_train) == 2400, len(targeted_train)
assert len(targeted_val) == 600, len(targeted_val)

print("DATA QA: PASS")
print("train:", len(train_df), train_df.label_id.value_counts().to_dict())
print("val:", len(val_df), val_df.label_id.value_counts().to_dict())
print("targeted train:", len(targeted_train))
print("targeted val:", len(targeted_val))


{
  "train": "/kaggle/input/datasets/ahmedabdelhedii/data-layer1-v2-1/preference_signal_train_v2_1.jsonl",
  "val": "/kaggle/input/datasets/ahmedabdelhedii/data-layer1-v2-1/preference_signal_val_v2_1.jsonl",
  "regression_val_v2": "/kaggle/input/datasets/ahmedabdelhedii/data-layer1-v2-1/preference_signal_regression_val_v2_5200.jsonl",
  "regression_holdout_v2": "/kaggle/input/datasets/ahmedabdelhedii/data-layer1-v2-1/preference_signal_regression_holdout_v2_1200.jsonl",
  "legacy_test": "/kaggle/input/datasets/ahmedabdelhedii/data-layer1-v2-1/preference_signal_legacy_test_v1_4000.jsonl"
}
DATA QA: PASS
train: 28400 {1: 14200, 0: 14200}
val: 5800 {0: 2900, 1: 2900}
targeted train: 2400
targeted val: 600


/tmp/ipykernel_58/1042551529.py:79: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_targeted_mask = train_df["is_targeted_v21"].fillna(False).astype(bool)
/tmp/ipykernel_58/1042551529.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val_targeted_mask = val_df["is_targeted_v21"].fillna(False).astype(bool)


In [16]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


class EncodedPreferenceDataset(Dataset):
    def __init__(self, encodings: dict[str, torch.Tensor], labels: torch.Tensor):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item


def encode_frame(frame: pd.DataFrame) -> EncodedPreferenceDataset:
    """
    Tokenize once in batches and keep CPU tensors in RAM.
    This removes repeated tokenizer work from every training epoch.
    """
    texts = frame["text"].astype(str).tolist()
    chunks = {}

    for start in range(0, len(texts), TOKENIZE_BATCH_SIZE):
        end = min(start + TOKENIZE_BATCH_SIZE, len(texts))

        encoded = tokenizer(
            texts[start:end],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )

        for key, value in encoded.items():
            chunks.setdefault(key, []).append(value)

        if end % (TOKENIZE_BATCH_SIZE * 4) == 0 or end == len(texts):
            print(f"tokenized {end}/{len(texts)}")

    encodings = {
        key: torch.cat(values, dim=0)
        for key, values in chunks.items()
    }

    labels = torch.tensor(
        frame["label_id"].astype(int).to_numpy(),
        dtype=torch.long,
    )

    return EncodedPreferenceDataset(
        encodings=encodings,
        labels=labels,
    )


print("Pre-tokenizing TRAIN...")
train_ds = encode_frame(train_df)

print("Pre-tokenizing VALIDATION...")
val_ds = encode_frame(val_df)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=generator,
)

val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model_core = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "NO_PREFERENCE_SIGNAL", 1: "PREFERENCE_SIGNAL"},
    label2id={"NO_PREFERENCE_SIGNAL": 0, "PREFERENCE_SIGNAL": 1},
).to(DEVICE)

if USE_MULTI_GPU:
    model = nn.DataParallel(
        model_core,
        device_ids=list(range(GPU_COUNT)),
        output_device=0,
    )
else:
    model = model_core


def unwrap_model(current_model):
    if isinstance(current_model, nn.DataParallel):
        return current_model.module

    return current_model


print("model ready")
print("model wrapper:", type(model).__name__)

if USE_MULTI_GPU:
    print("DataParallel device_ids:", model.device_ids)
    assert len(model.device_ids) >= 2


Pre-tokenizing TRAIN...
tokenized 8192/28400
tokenized 16384/28400
tokenized 24576/28400
tokenized 28400/28400
Pre-tokenizing VALIDATION...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenized 5800/5800
model ready
model wrapper: DataParallel
DataParallel device_ids: [0, 1]


In [17]:
# Optional but useful: verify both GPUs are visible before training.
if GPU_COUNT >= 2:
    print("Both Kaggle GPUs are visible.")
    print("GPU 0:", torch.cuda.get_device_name(0))
    print("GPU 1:", torch.cuda.get_device_name(1))
    print("DataParallel will split each training batch across GPU 0 and GPU 1.")
else:
    print(
        f"Only {GPU_COUNT} GPU detected. "
        "In Kaggle Settings, choose a dual-GPU accelerator if available."
    )


Both Kaggle GPUs are visible.
GPU 0: Tesla T4
GPU 1: Tesla T4
DataParallel will split each training batch across GPU 0 and GPU 1.


In [18]:
def binary_metrics(y_true, y_pred, probs=None):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    mp, mr, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "positive_precision": float(p),
        "positive_recall": float(r),
        "positive_f1": float(f1),
        "macro_precision": float(mp),
        "macro_recall": float(mr),
        "macro_f1": float(mf1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) else 0.0,
    }

    if probs is not None and len(set(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, probs))
        result["average_precision"] = float(
            average_precision_score(y_true, probs)
        )

    return result

@torch.no_grad()
def predict_loader(current_model, loader):
    current_model.eval()
    all_labels = []
    all_probs = []

    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        logits = current_model(**batch).logits
        probs = torch.softmax(logits, dim=-1)[:, 1]

        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.detach().cpu().numpy().tolist())

    return np.asarray(all_labels, dtype=int), np.asarray(all_probs, dtype=float)

def metrics_at_threshold(y, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return binary_metrics(y, pred, probs)


In [19]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

total_steps = (
    steps_per_epoch
    * MAX_EPOCHS
)

warmup_steps = int(
    total_steps
    * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available(),
)

if BEST_DIR.exists():
    shutil.rmtree(BEST_DIR)

best_macro_f1 = -1.0
best_epoch = None
epochs_without_improvement = 0
history = []

print("optimizer steps/epoch:", steps_per_epoch)
print("total optimizer steps:", total_steps)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    started = time.time()

    for step, batch in enumerate(train_loader, start=1):
        labels = batch.pop("labels").to(
            DEVICE,
            non_blocking=True,
        )

        batch = {
            key: value.to(
                DEVICE,
                non_blocking=True,
            )
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available(),
        ):
            outputs = model(
                **batch,
                labels=labels,
            )

            # IMPORTANT:
            # DataParallel returns one scalar loss per GPU, gathered on GPU 0.
            # Mean them before backward. On one GPU this is already scalar.
            raw_loss = outputs.loss

            if raw_loss.ndim > 0:
                raw_loss = raw_loss.mean()

            loss = (
                raw_loss
                / GRAD_ACCUM
            )

        scaler.scale(loss).backward()

        running_loss += (
            float(raw_loss.detach().item())
        )

        should_step = (
            step % GRAD_ACCUM == 0
            or step == len(train_loader)
        )

        if should_step:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    y_val, p_val = predict_loader(
        model,
        val_loader,
    )

    val_metrics_05 = metrics_at_threshold(
        y_val,
        p_val,
        0.5,
    )

    epoch_row = {
        "epoch": epoch,
        "train_loss": (
            running_loss
            / len(train_loader)
        ),
        "validation_at_0_5": val_metrics_05,
        "seconds": time.time() - started,
        "gpu_count": GPU_COUNT,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "effective_batch_size": (
            TRAIN_BATCH_SIZE
            * GRAD_ACCUM
        ),
    }

    history.append(epoch_row)

    print(
        json.dumps(
            epoch_row,
            indent=2,
        )
    )

    current = (
        val_metrics_05[
            "macro_f1"
        ]
    )

    if current > best_macro_f1 + 1e-12:
        best_macro_f1 = current
        best_epoch = epoch
        epochs_without_improvement = 0

        unwrap_model(
            model
        ).save_pretrained(
            BEST_DIR,
            safe_serialization=True,
        )

        tokenizer.save_pretrained(
            BEST_DIR / "tokenizer"
        )

        print(
            "NEW BEST:",
            best_epoch,
            best_macro_f1,
        )

    else:
        epochs_without_improvement += 1

        print(
            "no improvement:",
            epochs_without_improvement,
            "/",
            PATIENCE,
        )

    if epochs_without_improvement >= PATIENCE:
        print("EARLY STOP")
        break

print("BEST EPOCH:", best_epoch)
print(
    "BEST VALIDATION MACRO-F1 @0.5:",
    best_macro_f1,
)


optimizer steps/epoch: 888
total optimizer steps: 4440


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "epoch": 1,
  "train_loss": 0.10515697062988807,
  "validation_at_0_5": {
    "accuracy": 0.9877586206896551,
    "positive_precision": 1.0,
    "positive_recall": 0.9755172413793104,
    "positive_f1": 0.9876069122010822,
    "macro_precision": 0.9880511612251768,
    "macro_recall": 0.9877586206896551,
    "macro_f1": 0.9877567860272998,
    "tn": 2900,
    "fp": 0,
    "fn": 71,
    "tp": 2829,
    "false_positive_rate": 0.0,
    "false_negative_rate": 0.024482758620689656,
    "roc_auc": 0.9999789536266349,
    "average_precision": 0.9999794286128528
  },
  "seconds": 179.44404006004333,
  "gpu_count": 2,
  "train_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "effective_batch_size": 32
}
NEW BEST: 1 0.9877567860272998


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "epoch": 2,
  "train_loss": 0.0012038251435434496,
  "validation_at_0_5": {
    "accuracy": 0.9608620689655173,
    "positive_precision": 1.0,
    "positive_recall": 0.9217241379310345,
    "positive_f1": 0.959267898797775,
    "macro_precision": 0.963703229932843,
    "macro_recall": 0.9608620689655172,
    "macro_f1": 0.9608020263857797,
    "tn": 2900,
    "fp": 0,
    "fn": 227,
    "tp": 2673,
    "false_positive_rate": 0.0,
    "false_negative_rate": 0.07827586206896552,
    "roc_auc": 0.9927248513674198,
    "average_precision": 0.9942571128317773
  },
  "seconds": 178.61101627349854,
  "gpu_count": 2,
  "train_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "effective_batch_size": 32
}
no improvement: 1 / 2


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "epoch": 3,
  "train_loss": 0.0010336378824200717,
  "validation_at_0_5": {
    "accuracy": 0.983448275862069,
    "positive_precision": 1.0,
    "positive_recall": 0.9668965517241379,
    "positive_f1": 0.9831697054698457,
    "macro_precision": 0.9839786381842457,
    "macro_recall": 0.983448275862069,
    "macro_f1": 0.9834437401161983,
    "tn": 2900,
    "fp": 0,
    "fn": 96,
    "tp": 2804,
    "false_positive_rate": 0.0,
    "false_negative_rate": 0.03310344827586207,
    "roc_auc": 0.9999535077288941,
    "average_precision": 0.9999541772004852
  },
  "seconds": 178.90314245224,
  "gpu_count": 2,
  "train_batch_size": 32,
  "gradient_accumulation_steps": 1,
  "effective_batch_size": 32
}
no improvement: 2 / 2
EARLY STOP
BEST EPOCH: 1
BEST VALIDATION MACRO-F1 @0.5: 0.9877567860272998


In [20]:
# Reload the best checkpoint before any threshold calibration.
model_core = AutoModelForSequenceClassification.from_pretrained(
    BEST_DIR
).to(DEVICE)

if USE_MULTI_GPU:
    model = nn.DataParallel(
        model_core,
        device_ids=list(range(GPU_COUNT)),
        output_device=0,
    )
else:
    model = model_core

y_val, p_val = predict_loader(
    model,
    val_loader,
)


def choose_exact_threshold(
    y_true,
    probs,
    target_recall=TARGET_RECALL,
):
    candidates = np.unique(
        np.concatenate(
            [
                np.asarray(
                    [0.0, 1.0],
                    dtype=float,
                ),
                probs.astype(float),
            ]
        )
    )

    rows = []

    for threshold in candidates:
        metrics = metrics_at_threshold(
            y_true,
            probs,
            float(threshold),
        )

        rows.append(
            {
                "threshold": float(threshold),
                **metrics,
            }
        )

    eligible = [
        row
        for row in rows
        if (
            row["positive_recall"]
            >= target_recall
        )
    ]

    if eligible:
        chosen = max(
            eligible,
            key=lambda row: (
                row["positive_precision"],
                row["positive_f1"],
                row["accuracy"],
                row["threshold"],
            ),
        )

        policy = (
            "raw_transformer_max_precision_subject_to_"
            f"recall>={target_recall}"
        )

    else:
        chosen = max(
            rows,
            key=lambda row: (
                row["positive_f1"],
                row["macro_f1"],
                row["accuracy"],
                row["threshold"],
            ),
        )

        policy = (
            "raw_transformer_fallback_max_f1"
        )

    return (
        chosen,
        policy,
        len(rows),
    )


chosen, threshold_policy, candidate_count = choose_exact_threshold(
    y_val,
    p_val,
    TARGET_RECALL,
)

RAW_THRESHOLD_CANDIDATE = float(
    chosen["threshold"]
)

print(
    "CHECKPOINT + RAW THRESHOLD CANDIDATE NOW FROZEN"
)
print("best_epoch:", best_epoch)
print(
    "raw threshold candidate:",
    RAW_THRESHOLD_CANDIDATE,
)
print("policy:", threshold_policy)
print(
    json.dumps(
        chosen,
        indent=2,
    )
)


CHECKPOINT + RAW THRESHOLD CANDIDATE NOW FROZEN
best_epoch: 1
raw threshold candidate: 0.002538749249652028
policy: raw_transformer_max_precision_subject_to_recall>=0.985
{
  "threshold": 0.002538749249652028,
  "accuracy": 0.9963793103448276,
  "positive_precision": 1.0,
  "positive_recall": 0.9927586206896551,
  "positive_f1": 0.9963661533137221,
  "macro_precision": 0.9964053406367682,
  "macro_recall": 0.9963793103448275,
  "macro_f1": 0.9963792628791597,
  "tn": 2900,
  "fp": 0,
  "fn": 21,
  "tp": 2879,
  "false_positive_rate": 0.0,
  "false_negative_rate": 0.007241379310344828,
  "roc_auc": 0.9999789536266349,
  "average_precision": 0.9999794286128528
}


In [21]:
# Detailed validation metrics and targeted-val metrics after freeze.
val_metrics = metrics_at_threshold(
    y_val,
    p_val,
    RAW_THRESHOLD_CANDIDATE,
)

targeted_mask = val_df["is_targeted_v21"].fillna(False).astype(bool).to_numpy()

targeted_val_metrics = metrics_at_threshold(
    y_val[targeted_mask],
    p_val[targeted_mask],
    RAW_THRESHOLD_CANDIDATE,
)

per_targeted_family = {}

if "hard_family" in val_df.columns:
    family_series = val_df.loc[targeted_mask, "hard_family"].astype(str).reset_index(drop=True)

    y_t = y_val[targeted_mask]
    p_t = p_val[targeted_mask]

    for family in sorted(family_series.unique()):
        mask = (family_series.to_numpy() == family)
        per_targeted_family[family] = {
            "n": int(mask.sum()),
            "metrics": metrics_at_threshold(
                y_t[mask],
                p_t[mask],
                RAW_THRESHOLD_CANDIDATE,
            ),
        }

print("VALIDATION V2.1")
print(json.dumps(val_metrics, indent=2))
print("\nTARGETED VALIDATION")
print(json.dumps(targeted_val_metrics, indent=2))
print("\nPER TARGETED FAMILY")
print(json.dumps(per_targeted_family, indent=2))


VALIDATION V2.1
{
  "accuracy": 0.9963793103448276,
  "positive_precision": 1.0,
  "positive_recall": 0.9927586206896551,
  "positive_f1": 0.9963661533137221,
  "macro_precision": 0.9964053406367682,
  "macro_recall": 0.9963793103448275,
  "macro_f1": 0.9963792628791597,
  "tn": 2900,
  "fp": 0,
  "fn": 21,
  "tp": 2879,
  "false_positive_rate": 0.0,
  "false_negative_rate": 0.007241379310344828,
  "roc_auc": 0.9999789536266349,
  "average_precision": 0.9999794286128528
}

TARGETED VALIDATION
{
  "accuracy": 0.9866666666666667,
  "positive_precision": 1.0,
  "positive_recall": 0.9733333333333334,
  "positive_f1": 0.9864864864864865,
  "macro_precision": 0.987012987012987,
  "macro_recall": 0.9866666666666667,
  "macro_f1": 0.9866642958748222,
  "tn": 300,
  "fp": 0,
  "fn": 8,
  "tp": 292,
  "false_positive_rate": 0.0,
  "false_negative_rate": 0.02666666666666667,
  "roc_auc": 0.9996777777777778,
  "average_precision": 0.999691920155412
}

PER TARGETED FAMILY
{
  "v21_long_context_capa

/tmp/ipykernel_58/758661894.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  targeted_mask = val_df["is_targeted_v21"].fillna(False).astype(bool).to_numpy()


In [22]:
def evaluate_frame(frame: pd.DataFrame):
    dataset = encode_frame(
        frame
    )

    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    y, p = predict_loader(
        model,
        loader,
    )

    return metrics_at_threshold(
        y,
        p,
        RAW_THRESHOLD_CANDIDATE,
    )


# These are REGRESSION sets only.
# They have already been inspected and are not independent evidence.
regression_val_v2_metrics = evaluate_frame(
    reg_val_df
)

regression_holdout_v2_metrics = evaluate_frame(
    reg_holdout_df
)

legacy_test_metrics = evaluate_frame(
    legacy_df
)

print("OLD V2 VALIDATION REGRESSION")
print(
    json.dumps(
        regression_val_v2_metrics,
        indent=2,
    )
)

print(
    "\nOLD V2 HOLDOUT REGRESSION"
)
print(
    json.dumps(
        regression_holdout_v2_metrics,
        indent=2,
    )
)

print(
    "\nLEGACY V1 4000 REGRESSION"
)
print(
    json.dumps(
        legacy_test_metrics,
        indent=2,
    )
)


tokenized 5200/5200
tokenized 1200/1200
tokenized 4000/4000
OLD V2 VALIDATION REGRESSION
{
  "accuracy": 0.9975,
  "positive_precision": 1.0,
  "positive_recall": 0.995,
  "positive_f1": 0.9974937343358395,
  "macro_precision": 0.9975124378109452,
  "macro_recall": 0.9975,
  "macro_f1": 0.9974999843749024,
  "tn": 2600,
  "fp": 0,
  "fn": 13,
  "tp": 2587,
  "false_positive_rate": 0.0,
  "false_negative_rate": 0.005,
  "roc_auc": 0.999996449704142,
  "average_precision": 0.9999964503069513
}

OLD V2 HOLDOUT REGRESSION
{
  "accuracy": 0.97,
  "positive_precision": 1.0,
  "positive_recall": 0.94,
  "positive_f1": 0.9690721649484536,
  "macro_precision": 0.9716981132075472,
  "macro_recall": 0.97,
  "macro_f1": 0.9699729756781104,
  "tn": 600,
  "fp": 0,
  "fn": 36,
  "tp": 564,
  "false_positive_rate": 0.0,
  "false_negative_rate": 0.06,
  "roc_auc": 1.0,
  "average_precision": 1.0
}

LEGACY V1 4000 REGRESSION
{
  "accuracy": 0.99975,
  "positive_precision": 0.9995002498750625,
  "positi

In [23]:
# Export V2.1 artifact.
if OUT.exists():
    shutil.rmtree(OUT)

(OUT / "model").mkdir(parents=True, exist_ok=True)
(OUT / "tokenizer").mkdir(parents=True, exist_ok=True)

unwrap_model(model).save_pretrained(
    OUT / "model",
    safe_serialization=True,
)
tokenizer.save_pretrained(
    OUT / "tokenizer"
)

labels = {
    "id2label": {
        "0": "NO_PREFERENCE_SIGNAL",
        "1": "PREFERENCE_SIGNAL",
    },
    "label2id": {
        "NO_PREFERENCE_SIGNAL": 0,
        "PREFERENCE_SIGNAL": 1,
    },
}

thresholds = {
    "preference_signal_probability_threshold": RAW_THRESHOLD_CANDIDATE,
    "policy": threshold_policy,
    "target_positive_recall": TARGET_RECALL,
    "calibrated_on": "preference_signal_val_v2_1",
    "status": (
        "RAW_TRANSFORMER_VALIDATION_CANDIDATE_ONLY; "
        "final production threshold must be recalibrated locally "
        "with Transformer + PreferenceContextGuard before fresh holdout."
    ),
}

training_config = {
    "version": "v2.1",
    "seed": SEED,
    "base_model": BASE_MODEL,
    "initialization": "fresh_base_model_not_v2_checkpoint",
    "max_length": MAX_LENGTH,
    "batch_size": TRAIN_BATCH_SIZE,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRAD_ACCUM,
    "gpu_count": GPU_COUNT,
    "multi_gpu": USE_MULTI_GPU,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "target_recall": TARGET_RECALL,
    "best_epoch": best_epoch,
    "best_validation_macro_f1_at_0_5": best_macro_f1,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "targeted_train_size": len(targeted_train),
    "targeted_val_size": len(targeted_val),
    "regression_val_v2_size": len(reg_val_df),
    "regression_holdout_v2_size": len(reg_holdout_df),
    "legacy_test_size": len(legacy_df),
    "num_workers": NUM_WORKERS,
}

dataset_protocol = {
    "train": "preference_signal_train_v2_1",
    "validation": "preference_signal_val_v2_1",
    "old_v2_validation": "regression only; already inspected",
    "old_v2_holdout": "regression only; already inspected",
    "legacy_test": "preference_signal_legacy_test_v1_4000",
    "known_500": "not used in this notebook; local regression only",
    "fresh_final_holdout_v3": (
        "NOT GENERATED / NOT LOADED / NOT EVALUATED in this notebook. "
        "Generate only after V2.1 checkpoint and FINAL GUARDED threshold are frozen."
    ),
    "threshold_protocol": (
        "Notebook freezes a raw Transformer threshold candidate on V2.1 validation. "
        "Final production threshold is calibrated locally on the final guarded pipeline."
    ),
}

files = {
    "labels.json": labels,
    "thresholds.json": thresholds,
    "training_config.json": training_config,
    "validation_metrics.json": val_metrics,
    "targeted_validation_metrics.json": targeted_val_metrics,
    "targeted_validation_per_family.json": per_targeted_family,
    "regression_val_v2_metrics.json": regression_val_v2_metrics,
    "regression_holdout_v2_metrics.json": regression_holdout_v2_metrics,
    "legacy_test_metrics.json": legacy_test_metrics,
    "training_history.json": history,
    "dataset_protocol.json": dataset_protocol,
}

for filename, payload in files.items():
    (OUT / filename).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUT.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(OUT.parent),
            )

print("FINAL ARTIFACT:", FINAL_ZIP)
print("SIZE MB:", FINAL_ZIP.stat().st_size / (1024**2))
print("\nIMPORTANT:")
print("Do not generate the fresh final holdout yet.")
print("First download this artifact and run local guarded calibration.")


FINAL ARTIFACT: /kaggle/working/preference_signal_detector_v2_1.zip
SIZE MB: 480.0574769973755

IMPORTANT:
Do not generate the fresh final holdout yet.
First download this artifact and run local guarded calibration.


## Download final V2.1 ZIP

This cell uses a relative `FileLink` from `/kaggle/working` to avoid the
absolute-path Jupyter proxy 404 issue.


In [24]:
import os
from pathlib import Path
from IPython.display import FileLink, display

WORKING_DIR = Path("/kaggle/working")
FINAL_ZIP_NAME = "preference_signal_detector_v2_1.zip"
FINAL_ZIP_PATH = WORKING_DIR / FINAL_ZIP_NAME

if not FINAL_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Final ZIP not found: {FINAL_ZIP_PATH}. "
        "Run the export cell above first."
    )

print(f"Final artifact ready: {FINAL_ZIP_PATH}")
print(f"Size: {FINAL_ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

os.chdir(WORKING_DIR)

display(
    FileLink(
        FINAL_ZIP_NAME,
        result_html_prefix="Download final V2.1 artifact: "
    )
)

print(
    "\nIf the clickable link is blocked by Kaggle, "
    "download /kaggle/working/preference_signal_detector_v2_1.zip "
    "from the Files/Output panel."
)


Final artifact ready: /kaggle/working/preference_signal_detector_v2_1.zip
Size: 480.06 MB


/kaggle/working/preference_signal_detector_v2_1.zip


If the clickable link is blocked by Kaggle, download /kaggle/working/preference_signal_detector_v2_1.zip from the Files/Output panel.
